# Math 5422 — Project 3: Index Correlation (Correlation Arbitrage Backtest)

This notebook:
- Loads the provided market data (SPX + 11 sector ETFs), implied vols, and the yield curve.
- Builds the initial **theta-weighted** long ETF / short SPX straddle portfolio (quantities fixed after day 1).
- Reprices straddles daily with updated spot/vol/rates and **delta-hedges daily**.
- Produces the required output table (and an extended table matching the sample results format).
- Computes **implied correlation (10/18/2021)** and **realized correlation (10/18–12/17/2021)**.

> Assumptions (per project Q&A): dividend = 0; theta-neutral only at inception; delta-hedge daily; ignore cash financing from hedge trades.


In [38]:
import pandas as pd
import numpy as np
import math
from pathlib import Path

DATA_BOOK = Path(r"Project 3 Data.xlsx")
YIELD_BOOK = Path(r"YieldCurveData.xlsx")
SAMPLE_BOOK = Path(r"Project3_SampleResults.xlsx")


## 1) Helper functions - Data Cleaning

In [39]:
def _make_unique(cols):
    seen = {}
    out = []
    for c in cols:
        if c in seen:
            seen[c] += 1
            out.append(f"{c}.{seen[c]}")
        else:
            seen[c] = 0
            out.append(c)
    return out

def clean_market_sheet(path: Path, sheet: str) -> pd.DataFrame:
    raw = pd.read_excel(path, sheet_name=sheet, header=3)
    header = _make_unique(raw.iloc[0].tolist())
    df = raw.iloc[1:].copy()
    df.columns = header
    df = df.rename(columns={header[0]: "Dates"})
    df["Dates"] = pd.to_datetime(df["Dates"])
    # numeric columns
    for c in df.columns:
        if c != "Dates":
            df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=["Dates"]).reset_index(drop=True)
    return df

def clean_yield_curve(path: Path) -> pd.DataFrame:
    # The yield curve sheet uses two header rows (instrument name + PX_LAST)
    df = pd.read_excel(path, sheet_name="Swap Curve", header=[2, 3])

    cols = []
    for a, b in df.columns:
        if "Unnamed" in str(a) and "Unnamed" in str(b):
            cols.append("Dates")
        else:
            cols.append(b)  # instrument name, e.g. 'US0003M Index'
    df.columns = _make_unique(cols)

    # First row is still a header row ('Dates', 'PX_LAST', ...)
    df = df.iloc[1:].copy()
    df["Dates"] = pd.to_datetime(df["Dates"])
    for c in df.columns:
        if c != "Dates":
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.dropna(subset=["Dates"]).reset_index(drop=True)
    return df

# Load sheets
SPX = clean_market_sheet(DATA_BOOK, "SPX")
ETFS = {t: clean_market_sheet(DATA_BOOK, t) for t in ["XLU","XLB","XLRE","XLE","XLP","XLI","XLC","XLF","XLY","XLV","XLK"]}
YC = clean_yield_curve(YIELD_BOOK)

print("SPX rows:", len(SPX), "cols:", len(SPX.columns))
print("YC rows:", len(YC), "cols:", len(YC.columns))
print("ETF example rows:", len(ETFS["XLK"]))
SPX.head()


SPX rows: 53 cols: 31
YC rows: 82 cols: 15
ETF example rows: 53


,Dates,PX_LAST,1M_CALL_IMP_VOL_10DELTA_DFLT,1M_CALL_IMP_VOL_25DELTA_DFLT,1M_CALL_IMP_VOL_40DELTA_DFLT,1M_CALL_IMP_VOL_50DELTA_DFLT,1M_CALL_IMP_VOL_60DELTA_DFLT,1M_CALL_IMP_VOL_75DELTA_DFLT,1M_CALL_IMP_VOL_90DELTA_DFLT,1M_PUT_IMP_VOL_10DELTA_DFLT,...,2M_CALL_IMP_VOL_75DELTA_DFLT,2M_CALL_IMP_VOL_90DELTA_DFLT,2M_PUT_IMP_VOL_10DELTA_DFLT,2M_PUT_IMP_VOL_25DELTA_DFLT,2M_PUT_IMP_VOL_40DELTA_DFLT,2M_PUT_IMP_VOL_50DELTA_DFLT,2M_PUT_IMP_VOL_60DELTA_DFLT,2M_PUT_IMP_VOL_60DELTA_DFLT.1,2M_PUT_IMP_VOL_75DELTA_DFLT,2M_PUT_IMP_VOL_90DELTA_DFLT
0,2021-10-18,4486.46,10.18,10.70,11.66,12.52,13.88,16.76,21.97,21.97,...,18.83,24.58,24.58,18.83,15.63,14.11,13.13,13.13,11.99,11.47
1,2021-10-19,4519.63,9.97,10.43,11.22,12.04,13.13,15.86,21.20,21.20,...,18.35,24.34,24.34,18.35,15.12,13.72,12.76,12.76,11.65,11.09
2,2021-10-20,4536.19,9.64,10.31,11.06,11.78,12.92,15.56,20.93,20.93,...,18.01,23.95,23.95,18.01,14.84,13.51,12.60,12.60,11.54,10.91
3,2021-10-21,4549.78,9.00,9.76,10.55,11.34,12.45,15.03,20.32,20.32,...,17.52,23.43,23.43,17.52,14.50,13.20,12.22,12.22,11.17,10.52
4,2021-10-22,4544.90,8.94,9.90,10.96,11.74,12.91,15.60,20.75,20.75,...,18.08,23.98,23.98,18.08,14.92,13.54,12.48,12.48,11.28,10.44


## 2) Sector weights

These are the sector weights used to allocate SPX straddle theta to sector ETF straddles.


In [40]:
WEIGHTS = {
    "XLK": 27.914488/100,  # Info Tech
    "XLV": 12.692172/100,  # Health Care
    "XLY": 12.618698/100,  # Cons Discretionary
    "XLF": 11.499232/100,  # Financials
    "XLC": 11.042944/100,  # Comm Services
    "XLI":  8.127506/100,  # Industrials
    "XLP":  5.670948/100,  # Cons Staples
    "XLE":  2.924949/100,  # Energy
    "XLRE": 2.579896/100,  # Real Estate
    "XLB":  2.527329/100,  # Materials
    "XLU":  2.401839/100,  # Utilities
}
print("Sum of weights:", sum(WEIGHTS.values()))

Sum of weights: 1.00000001


## 3) Black–Scholes pricing + delta for ATM straddles

We will price an ATM straddle each day using:
- Spot `S` from `PX_LAST`
- Volatility from a single column (by default `1M_CALL_IMP_VOL_50DELTA_DFLT`)
- Risk-free rate from `US0003M Index` (3M) in the yield curve sheet
- Dividend yield `q = 0` (per Q&A)
- Strike `K = S` (ATM by spot)


In [41]:
def norm_cdf(x: float) -> float:
    return 0.5*(1.0 + math.erf(x/math.sqrt(2.0)))

def bs_price(S: float, K: float, T: float, r: float, sigma: float, cp: int) -> float:
    # cp = +1 call, -1 put
    if T <= 0 or sigma <= 0:
        return max(S-K, 0.0) if cp == 1 else max(K-S, 0.0)
    d1 = (math.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*math.sqrt(T))
    d2 = d1 - sigma*math.sqrt(T)
    if cp == 1:
        return S*norm_cdf(d1) - K*math.exp(-r*T)*norm_cdf(d2)
    else:
        return K*math.exp(-r*T)*norm_cdf(-d2) - S*norm_cdf(-d1)

def bs_delta(S: float, K: float, T: float, r: float, sigma: float, cp: int) -> float:
    if T <= 0 or sigma <= 0:
        if cp == 1:
            return 1.0 if S > K else 0.0
        else:
            return -1.0 if S < K else 0.0
    d1 = (math.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*math.sqrt(T))
    Nd1 = norm_cdf(d1)
    return Nd1 if cp == 1 else Nd1 - 1.0

def straddle_price_delta(S: float, T: float, r: float, sigma: float) -> tuple[float, float]:
    K = S  # spot ATM
    call = bs_price(S, K, T, r, sigma, +1)
    put  = bs_price(S, K, T, r, sigma, -1)
    price = call + put
    delta = bs_delta(S, K, T, r, sigma, +1) + bs_delta(S, K, T, r, sigma, -1)
    return price, delta

def get_rate_3m(date: pd.Timestamp) -> float:
    # use US0003M Index (percent) -> decimal
    if "US0003M Index" not in YC.columns:
        raise KeyError("US0003M Index not found in yield curve columns.")
    row = YC.loc[YC["Dates"] == date]
    if row.empty:
        # if date missing (shouldn't), use last available before date
        idx = YC["Dates"].searchsorted(date) - 1
        idx = max(idx, 0)
        val = float(YC.iloc[idx]["US0003M Index"])
    else:
        val = float(row.iloc[0]["US0003M Index"])
    return val / 100.0


## 4) Portfolio construction (theta-weighted)

In [42]:
START_DATE = pd.Timestamp("2021-10-18")
END_DATE   = pd.Timestamp("2021-12-17")

# Dates for the required output rows come from the template "Output table" sheet
OUTPUT_TEMPLATE = pd.read_excel(DATA_BOOK, sheet_name="Output table")
DATES = pd.to_datetime(OUTPUT_TEMPLATE["Dates"].dropna()).tolist()

VOL_COL = "1M_CALL_IMP_VOL_50DELTA_DFLT"  # per project simplification

qty_spx = -1000.0  # short 1000 straddles

def time_to_maturity(d: pd.Timestamp, daycount: int = 365) -> float:
    return max((END_DATE - d).days / daycount, 0.0)

def get_sigma(df: pd.DataFrame, date: pd.Timestamp, vol_col: str = VOL_COL) -> float:
    row = df.loc[df["Dates"] == date]
    if row.empty:
        raise KeyError(f"Missing date {date} in data")
    return float(row.iloc[0][vol_col]) / 100.0

def get_spot(df: pd.DataFrame, date: pd.Timestamp) -> float:
    row = df.loc[df["Dates"] == date]
    if row.empty:
        raise KeyError(f"Missing date {date} in data")
    return float(row.iloc[0]["PX_LAST"])

# Initial premiums on 10/18
T0 = time_to_maturity(START_DATE, 365)
r0 = get_rate_3m(START_DATE)

S_spx0 = get_spot(SPX, START_DATE)
sig_spx0 = get_sigma(SPX, START_DATE, VOL_COL)
prem_spx0, delta_spx0 = straddle_price_delta(S_spx0, T0, r0, sig_spx0)

etf_qty = {}
etf_prem0 = {}
for t, df in ETFS.items():
    S0 = get_spot(df, START_DATE)
    sig0 = get_sigma(df, START_DATE, VOL_COL)
    prem0, _ = straddle_price_delta(S0, T0, r0, sig0)
    etf_prem0[t] = prem0
    etf_qty[t] = (abs(qty_spx) * prem_spx0 * WEIGHTS[t]) / prem0  # long

pd.Series(etf_qty).sort_values(ascending=False)


XLF     9254.338629
XLK     6457.631797
XLC     5120.488019
XLV     3844.829319
XLP     3708.202855
XLI     2983.230331
XLY     2331.093311
XLRE    2085.992633
XLU     1401.643258
XLE     1069.796593
XLB      876.835103
dtype: float64

## 5) Backtest: daily repricing + daily delta hedge

PnL conventions:
- **Option (straddle) PnL**: `qty * (V_t - V_{t-1})`
- **Hedge PnL**: `shares_{t-1} * (S_t - S_{t-1})`

Delta hedge shares:
- We set hedge shares as `shares = qty * delta`.

After computing day *t* PnL, we rebalance shares to be delta-neutral for day *t+1*.


In [43]:
def run_backtest(daycount: int = 365, vol_col: str = VOL_COL) -> pd.DataFrame:
    # Initial state at START_DATE
    T = time_to_maturity(START_DATE, daycount)
    r = get_rate_3m(START_DATE)

    S_spx = get_spot(SPX, START_DATE)
    sig_spx = get_sigma(SPX, START_DATE, vol_col)
    V_spx, d_spx = straddle_price_delta(S_spx, T, r, sig_spx)

    spx_shares = qty_spx * d_spx  # hedge shares

    etf_state = {}
    for t, df in ETFS.items():
        S = get_spot(df, START_DATE)
        sig = get_sigma(df, START_DATE, vol_col)
        V, d = straddle_price_delta(S, T, r, sig)
        etf_state[t] = {"S": S, "V": V, "shares": etf_qty[t] * d}

    spx_prev = {"S": S_spx, "V": V_spx, "shares": spx_shares}

    rows = []
    for d in DATES:
        T = time_to_maturity(d, daycount)
        r = get_rate_3m(d)

        # --- SPX leg ---
        S = get_spot(SPX, d)
        sig = get_sigma(SPX, d, vol_col)
        V, delt = straddle_price_delta(S, T, r, sig)

        spx_straddle_pnl = qty_spx * (V - spx_prev["V"])
        spx_hedge_pnl    = spx_prev["shares"] * (S - spx_prev["S"])
        spx_daily        = spx_straddle_pnl + spx_hedge_pnl

        # rebalance hedge
        spx_prev = {"S": S, "V": V, "shares": qty_spx * delt}

        # --- ETF legs (aggregate) ---
        etf_straddle_pnl = 0.0
        etf_hedge_pnl = 0.0

        for t, df in ETFS.items():
            S_t = get_spot(df, d)
            sig_t = get_sigma(df, d, vol_col)
            V_t, delt_t = straddle_price_delta(S_t, T, r, sig_t)

            prev = etf_state[t]
            qty = etf_qty[t]

            etf_straddle_pnl += qty * (V_t - prev["V"])
            etf_hedge_pnl    += prev["shares"] * (S_t - prev["S"])

            # rebalance
            etf_state[t] = {"S": S_t, "V": V_t, "shares": qty * delt_t}

        etf_daily = etf_straddle_pnl + etf_hedge_pnl
        total = spx_daily + etf_daily

        rows.append([d, spx_straddle_pnl, spx_hedge_pnl, spx_daily,
                     etf_straddle_pnl, etf_hedge_pnl, etf_daily, total])

    out = pd.DataFrame(rows, columns=[
        "Dates",
        "SPX Straddle PnL", "SPX Hedge PnL", "SPX Daily PnL",
        "ETF Straddle PnL", "ETF Hedge PnL", "ETF Daily PnL",
        "Total PnL"
    ])
    return out

results = run_backtest(daycount=365, vol_col=VOL_COL)
results.head()


,Dates,SPX Straddle PnL,SPX Hedge PnL,SPX Daily PnL,ETF Straddle PnL,ETF Hedge PnL,ETF Daily PnL,Total PnL
0,2021-10-18,-0.000000,-0.000000,-0.000000,0.000000,0.000000,0.000000,0.000000
1,2021-10-19,7143.859894,-784.309546,6359.550348,-2032.297208,771.429796,-1260.867412,5098.682936
2,2021-10-20,4600.374786,-376.885171,4223.489615,-7404.140755,395.398059,-7008.742696,-2785.253081
3,2021-10-21,7275.364789,-301.610322,6973.754467,-5119.211820,276.175741,-4843.036079,2130.718388
4,2021-10-22,-4073.946425,104.040178,-3969.906246,8443.336645,-39.892885,8403.443759,4433.537513


## 6) Save required output tables

- `Project3_OutputTable.csv` — matches the template columns
- `Project3_ExtendedResults.csv` — matches the sample-results style columns
- `Project3_Output.xlsx` — both tabs in one Excel file


In [46]:
# Template-style output (as required by PDF)
required_cols = ["Dates","SPX Straddle PnL","SPX Hedge PnL","ETF Straddle PnL","ETF Hedge PnL","Total PnL"]
required_out = results[required_cols].copy()

out_csv = Path("Project3_OutputTable.csv")
ext_csv = Path("Project3_ExtendedResults.csv")
out_xlsx = Path("Project3_OutputFinal.xlsx")

with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    required_out.to_excel(writer, sheet_name="OutputTable", index=False)
    results.to_excel(writer, sheet_name="ExtendedResults", index=False)

out_csv, ext_csv, out_xlsx


(WindowsPath('Project3_OutputTable.csv'),
 WindowsPath('Project3_ExtendedResults.csv'),
 WindowsPath('Project3_OutputFinal.xlsx'))

## 8) Implied vs Realized correlation

We compute:
- **Implied correlation** at 10/18/2021 from the option-implied vols (ATM) using a 1-factor (single-ρ) approximation:
\[
\sigma_{index}^2 = \sum_i w_i^2\sigma_i^2 + \rho \sum_{i\neq j} w_iw_j\sigma_i\sigma_j
\]
Solve for ρ.

- **Realized correlation** from realized vol/cov of daily log returns over 10/18–12/17 and the same 1-factor approximation.


In [45]:
def implied_rho_single_factor(sig_index: float, sigs: dict[str,float], weights: dict[str,float]) -> float:
    w = weights
    a = sum((w[t]**2)*(sigs[t]**2) for t in sigs)
    b = 0.0
    keys = list(sigs.keys())
    for i in range(len(keys)):
        for j in range(len(keys)):
            if i == j:
                continue
            ti, tj = keys[i], keys[j]
            b += w[ti]*w[tj]*sigs[ti]*sigs[tj]
    # If b is ~0, correlation not identifiable
    return (sig_index**2 - a) / b

# --- Implied correlation on 10/18 ---
sig_index_imp = get_sigma(SPX, START_DATE, VOL_COL)
sig_etf_imp = {t: get_sigma(df, START_DATE, VOL_COL) for t, df in ETFS.items()}
rho_impl = implied_rho_single_factor(sig_index_imp, sig_etf_imp, WEIGHTS)

# --- Realized correlation over the window ---
# Use daily log returns for each ETF, compute annualized vol/cov with 252.
rets = pd.DataFrame({"Dates": SPX["Dates"], "SPX": np.log(SPX["PX_LAST"]).diff()}).dropna()
for t, df in ETFS.items():
    tmp = pd.DataFrame({"Dates": df["Dates"], t: np.log(df["PX_LAST"]).diff()}).dropna()
    rets = rets.merge(tmp, on="Dates", how="inner")

# restrict window
rets = rets[(rets["Dates"] >= START_DATE) & (rets["Dates"] <= END_DATE)].copy()

# realized vols for ETFs and index (annualized)
ann = 252
sig_index_real = float(rets["SPX"].std(ddof=1) * math.sqrt(ann))
sig_etf_real = {t: float(rets[t].std(ddof=1) * math.sqrt(ann)) for t in ETFS.keys()}

# realized covariance term for basket under single-factor approximation
rho_real = implied_rho_single_factor(sig_index_real, sig_etf_real, WEIGHTS)

print("Implied correlation (single-ρ) on 10/18:", rho_impl)
print("Realized correlation (single-ρ) over window:", rho_real)


Implied correlation (single-ρ) on 10/18: 0.5628343649903168
Realized correlation (single-ρ) over window: 0.5250095995495805
